In [ ]:
%pip install llama-index
%pip install llama-index-llms-ollama
%pip install llama-index-embeddings-huggingface
%pip install llama-index-core llama-index-readers-file 
%pip install -e llama-index-integrations/llms/llama-index-llms-ollama
%pip install llama-index-llms-replicate
%pip install llama-parse
%pip install replicate

In [ ]:
%pip install beir

In [ ]:
%pip install spacy

In [ ]:
%pip install llama-index llama-index-llms-huggingface llama-index-embeddings-huggingface transformers accelerate bitsandbytes llama-index-readers-web

Making a cache directory to store cache files 


In [ ]:
import os
os.environ['LLAMA_INDEX_CACHE_DIR'] = './h'

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama


def get_meta(file_path):
    return {"foo": "bar", "file_path": file_path}

reader = SimpleDirectoryReader("./data",recursive = True,file_metadata=get_meta)
#SimpleDirectoryReader(input_files=["path/to/file1", "path/to/file2"])
#SimpleDirectoryReader(
   # input_dir="path/to/directory", exclude=["path/to/file1", "path/to/file2"]
#)
#SimpleDirectoryReader(
   # input_dir="path/to/directory", required_exts=[".pdf", ".docx"]
#)
#SimpleDirectoryReader(input_dir="path/to/directory", num_files_limit=100)
all_docs = []
for docs in reader.iter_data():
    # <do something with the documents per file>
    all_docs.extend(docs)
"""
from llama_index.core import SimpleDirectoryReader
from llama_index.core.readers.base import BaseReader
from llama_index.core import Document


class MyFileReader(BaseReader):
    def load_data(self, file, extra_info=None):
        with open(file, "r") as f:
            text = f.read()
        # load_data returns a list of Document objects
        return [Document(text=text + "Foobar", extra_info=extra_info or {})]


reader = SimpleDirectoryReader(
    input_dir="./data", file_extractor={".myfile": MyFileReader()}
)

"""
documents = reader.load_data()
# Configure settings
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")
Settings.llm = Ollama(model="llama3", request_timeout=360.0, temperature=0.1)

# Create index
index = VectorStoreIndex.from_documents(documents)


In [ ]:
"""
from llama_index.core import VectorStoreIndex, download_loader

from llama_index.readers.google import GoogleDocsReader

gdoc_ids = ["1wf-y2pd9C878Oh-FmLH7Q_BQkljdm6TQal-c1pUfrec"]
loader = GoogleDocsReader()
documents = loader.load_data(document_ids=gdoc_ids)
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
query_engine.query("Where did the author go to school?")
"""

In [ ]:
documents

In [ ]:
query_engine1 = index.as_query_engine(response_mode="compact")
response = query_engine1.query("What are the things in the Sprint planning checklist? and also list the page number")
print(response)

In [ ]:
query_engine1 = index.as_query_engine(response_mode="refine")
response = query_engine1.query("Explain the error details and also list the page number")
print(response)

In [ ]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)
from llama_index.core.indices import SummaryIndex

summary_index = SummaryIndex.from_documents(documents)

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

vector_tool = QueryEngineTool(
    vector_index.as_query_engine(),
    metadata=ToolMetadata(
        name="vector_search",
        description="Useful for searching for specific facts."
    )
)

summary_tool = QueryEngineTool(
    summary_index.as_query_engine(response_mode="tree_summarize"),
    metadata=ToolMetadata(
        name="summary",
        description="Useful for summarizing an entire document."
    )
)

In [ ]:

from llama_index.core.query_engine import RouterQueryEngine

query_engine = RouterQueryEngine.from_defaults(
    [vector_tool, summary_tool],
    select_multi=True
)
query = "What are the things to be kept in mind before starting the meet?"
response = query_engine.query("What are the things to be kept in mind before starting the meet?")

print(response)

### Generating questions

In [ ]:

from llama_index.core.evaluation import DatasetGenerator, RelevancyEvaluator
data_generator = DatasetGenerator.from_documents(documents)

In [ ]:
eval_questions = data_generator.generate_questions_from_nodes()

In [ ]:
eval_questions

## Trial evaluation: didn't work

In [ ]:
evaluator = RelevancyEvaluator(llm= Settings.llm)

In [ ]:
from llama_index.core import Response
import pandas as pd

# Function to display evaluation results in a DataFrame
def display_eval_df(query: str, response: Response, eval_result) -> None:
    # Ensure eval_result is a list, even if it contains a single item
    if not isinstance(eval_result, list):
        eval_result = [eval_result]

    # Creating DataFrame
    eval_df = pd.DataFrame({
        "Query": [query] * len(eval_result),
        "Response": [str(response)] * len(eval_result),
        "Source": [response.source_nodes[0].node.get_content()[:1000] + "..."] * len(eval_result),
        "Evaluation Result": eval_result,
    })

    # Styling DataFrame
    eval_df = eval_df.style.set_properties(
        **{
            "inline-size": "0px",
            "overflow-wrap": "break-word",
        },
        subset=["Response", "Source"]
    )
    display(eval_df)

# Example usage
response_vector = query_engine.query(eval_questions[2])
eval_result = evaluator.evaluate_response(
    query=eval_questions[2], response=response_vector
)
display_eval_df(eval_questions[2], response_vector, eval_result)


### This is done so that in future: we can automate the ground truth part

In [ ]:
from llama_index.core import Document
all_text = ""

for doc in documents:
    all_text += doc.text

giant_document = Document(text=all_text)

In [ ]:
giant_document

## Generating questions

In [ ]:
import os
import random
random.seed(42)

from llama_index.core import ServiceContext
from llama_index.core.prompts import Prompt
from llama_index.core.evaluation import DatasetGenerator

question_dataset = []
if os.path.exists("questions.txt"):
    with open("questions.txt", "r") as f:
        for line in f:
            question_dataset.append(line.strip())
else:
    # generate questions
    data_generator = DatasetGenerator.from_documents(
        [giant_document],
        text_question_template=Prompt(
            "A sample from the LlamaIndex documentation is below.\n"
            "---------------------\n"
            "{context_str}\n"
            "---------------------\n"
            "Using the documentation sample, carefully follow the instructions below:\n"
            "{query_str}"
        ),
        question_gen_query=(
            "You are an evaluator for a search pipeline. Your task is to write a single question "
            "using the provided documentation sample above to test the search pipeline. The question should "
            "reference specific names, functions, and terms. Restrict the question to the "
            "context information provided.\n"
            "Question: "
        ),
       
    )
    generated_questions = data_generator.generate_questions_from_nodes()

    # randomly pick 40 questions from each dataset
    generated_questions = random.sample(generated_questions, 40)
    question_dataset.extend(generated_questions)

    print(f"Generated {len(question_dataset)} questions.")

In [ ]:
print(random.sample(question_dataset, 5))


## Taking two text files : questions.txt and ground_truth.txt
## We generate responses for all questions in questions.txt.
## We compare these responses with ground_truth.txt
## To find hallucination score of each
## and therefore average hallucination score = Accuracy

In [ ]:
from sentence_transformers import CrossEncoder 
model = CrossEncoder("vectara/hallucination_evaluation_model")


In [ ]:
def load_questions(file_path):
    with open(file_path, 'r') as file:
        questions = file.readlines()
    questions = [q.strip() for q in questions]  # Remove any leading/trailing whitespace
    return questions


In [ ]:
questions = load_questions("questions.txt")

In [ ]:
# Evaluate the model with the questions
def evaluate_model(questions):
    responses = []
    for question in questions:
        response = query_engine.query(question)
          # Assuming `model` is a callable that takes a question and returns a response
        responses.append(response)
    return responses

In [ ]:
responses = evaluate_model(questions)

In [ ]:

responses

In [ ]:
responses[0].response

In [ ]:
n = 0
res = []
while n<10:
    res.append(responses[n].response)
    n += 1


In [ ]:
def load_ground_truth(file_path):
    with open(file_path, 'r') as file:
        truth = file.readlines()
        truth = [t.strip() for t in truth]  # Remove any leading/trailing whitespace'
        print(truth)
    return truth


In [ ]:
gt = load_ground_truth("ground_truth.txt")

In [ ]:
gt

In [ ]:
def evaluation(questions,res,n):
    sentence_pairs = []
    for question in questions:
        while n<10:
            sentence_pairs.append((questions[n], res[n]))
            n += 1
    return sentence_pairs

In [ ]:
res

In [ ]:
sentence_pairs = evaluation(gt,res,0)

In [ ]:
sentence_pairs

In [ ]:
score  = model.predict(sentence_pairs)
print(f"Hallucination score: {score}\n")

In [ ]:
s = 0
n=10
for i in score:
    s = s + i
print("Accuracy of the 10 responses:",s/n * 100)

## A normal llama without RAG


In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
from llama_index.llms.replicate import Replicate

llm_replicate = Replicate(model="meta/meta-llama-3-8b-instruct")

In [ ]:
llm = Ollama(model="llama3", request_timeout=120.0)
response = llm.complete("What is in the sprint planning checklist?")

print(response)

## Comparing between compact, refine and tree-summarise methods.


In [ ]:
query_engine = vector_index.as_query_engine(response_mode="compact")

response = query_engine.query("Summarise both the templates")

print(response)

In [ ]:
query_engine = vector_index.as_query_engine(response_mode="refine")

response = query_engine.query("Summarise both the templates")

print(response)

In [ ]:
query_engine = vector_index.as_query_engine(response_mode="tree_summarize")

response = query_engine.query("Summarise both the templates")

print(response)

## Making an agent

In [ ]:
from llama_index.core.agent import ReActAgent

agent = ReActAgent.from_tools(
    [vector_tool, summary_tool],
    llm=llm,
    verbose=True
)

In [ ]:
response = agent.chat("What is the difference in the 2 templates?")
print(response)

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

text_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=10)

# global
from llama_index.core import Settings

Settings.text_splitter = text_splitter

# per-index
index = VectorStoreIndex.from_documents(
    documents, transformations=[text_splitter]
)
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import TokenTextSplitter

pipeline = IngestionPipeline(transformations=[TokenTextSplitter(), ...])

nodes = pipeline.run(documents=documents)